Wczytanie danych

In [83]:
import os, json, time, numpy as np, pandas as pd, dash
import plotly.express as px, plotly.graph_objs as go, dash_bootstrap_components as dbc
from dash import Input, Output, State, dcc, html, dash_table, ALL, MATCH, ctx
#region Start
app = dash.Dash(
    __name__,
    external_stylesheets=[
        dbc.themes.BOOTSTRAP, #Wygląd
        "https://cdn.jsdelivr.net/npm/bootstrap-icons@1.11.0/font/bootstrap-icons.css"
    ],
    suppress_callback_exceptions=True
)
DATA_FOLDER = 'Dane Dash/' #Folder do zczytania gotowych danych
frames = []  
for filename in os.listdir(DATA_FOLDER):
    if filename.endswith('.json'):
        filepath = os.path.join(DATA_FOLDER, filename)
        frames.append(pd.read_json(filepath))
df = pd.concat(frames, ignore_index=True)

Kolumny z dniem tygodnia i godziną

In [279]:
# Zmiana na czas polski
timestamp = df['timestamp']
timestamp_pl = timestamp.dt.tz_convert('Europe/Warsaw')
df['timestamp_pl'] = timestamp_pl
df['day_of_year'] = timestamp_pl.dt.day_of_year
df['weekday'] = timestamp_pl.dt.day_name('Pl')
df['weekday_number'] = timestamp_pl.dt.weekday + 1
df['hour'] = timestamp_pl.dt.hour
df.sample(2)

,timestamp,duration,app,title,status,category,osoba,device,timestamp_pl,weekday,weekday_number,hour,day_of_year
27728,2025-12-30 12:39:58.749000+00:00,1.008,Spotify.exe,Spotify Premium,not-afk,muzyka,A,PC,2025-12-30 13:39:58.749000+01:00,Wtorek,2,13,364
4738,2025-12-23 11:52:04.860000+00:00,8.059,chrome.exe,,not-afk,studia,AK,PC,2025-12-23 12:52:04.860000+01:00,Wtorek,2,12,357


# Średnia aktywność w dniach tygodnia

In [280]:
df2 = df.groupby(["osoba","weekday","weekday_number","category","day_of_year"])['duration'].aggregate(total_activity = np.sum).reset_index()
df2 = df2.groupby(["osoba","weekday","weekday_number","category"])['total_activity'].aggregate(average_activity = np.mean).reset_index()
df2['average_activity'] /= 3600
df2.head()

C:\Users\Antek\AppData\Local\Temp\ipykernel_5392\3375787278.py:1: FutureWarning:

The provided callable <function sum at 0x000001EEA8C4EF20> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.

C:\Users\Antek\AppData\Local\Temp\ipykernel_5392\3375787278.py:2: FutureWarning:

The provided callable <function mean at 0x000001EEA8C5C360> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.



,osoba,weekday,weekday_number,category,average_activity
0,A,Czwartek,4,Komunikatory,0.089587
1,A,Czwartek,4,gry komputerowe,3.665453
2,A,Czwartek,4,inne,0.700383
3,A,Czwartek,4,muzyka,0.029896
4,A,Czwartek,4,nauka,0.717976


In [ ]:
# To będzie do czegoś innego
Suma_z_dnia = df.groupby(['osoba','day_of_year'])['duration'].sum().reset_index().sort_values('duration',ascending=False)
Suma_z_dnia['duration'] /= 3600
Suma_z_dnia

,osoba,day_of_year,duration
5,A,349,14.402186
16,A,360,12.646857
17,A,361,12.493067
43,AK,365,12.306290


In [284]:
osoba = "AK"
df3 = df2[df2['osoba'] == osoba]
ylim = np.ceil(df2.groupby(['osoba','weekday'])['average_activity'].sum().reset_index()['average_activity'].max())
px.bar(df3,
       x='weekday',
       y = 'average_activity',
       color='category',
       category_orders={'weekday':['Poniedziałek','Wtorek','Środa','Czwartek','Piątek','Sobota','Niedziela']},
       labels={'weekday_number':'dzień tygodnia','average_activity':'średnia aktywność (godz.)','category':'kategoria'},
       title=f"Średnia aktywność w dniach tygodnia: {osoba}",
       template='plotly_white',
       range_y=[0,ylim])